# Phase 2: Data Quality & Missingness Audit

## Project: Credit Risk Modelling & Independent Model Validation (SR 11-7)

### Notebook Objectives
1. Perform missing value audit across all candidate risk features.
2. Evaluate feature outlier distributions using IQR bounds.
3. Apply 1st/99th percentile Winsorization to extreme financial tails.
4. Verify dataset integrity for downstream feature selection.

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = root_path / "src"
for p in [str(root_path), str(src_path)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Environment & core risk libraries initialized successfully!")

Environment & core risk libraries initialized successfully!


In [2]:
# Data Loading Helper with Synthetic Fallback
data_file = root_path / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
if data_file.is_file():
    df = pd.read_csv(data_file, nrows=50000, low_memory=False)
    bad = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
    good = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
    df["target"] = np.nan
    df.loc[df["loan_status"].isin(bad), "target"] = 1.0
    df.loc[df["loan_status"].isin(good), "target"] = 0.0
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
else:
    df = mock_df.copy()

print(f"Dataset Population Loaded: {len(df):,} loans | Default Rate: {df['target'].mean():.4%}")

Dataset Population Loaded: 44,252 loans | Default Rate: 20.9572%


In [3]:
missing_series = df.isnull().mean() * 100
missing_df = pd.DataFrame({"Feature": missing_series.index, "Missing_Pct": missing_series.values})
missing_df = missing_df.sort_values("Missing_Pct", ascending=False).head(15)
missing_df

,Feature,Missing_Pct
1,member_id,100.000000
121,sec_app_open_acc,100.000000
122,sec_app_revol_util,100.000000
115,revol_bal_joint,100.000000
124,sec_app_num_rev_accts,100.000000
125,sec_app_chargeoff_within_12_mths,100.000000
126,sec_app_collections_12_mths_ex_med,100.000000
119,sec_app_inq_last_6mths,100.000000
116,sec_app_fico_range_low,100.000000
117,sec_app_fico_range_high,100.000000
